In [ ]:
# Test PyTorch DataLoader
try:
    from data_engine.data_loader import BDD100KDataset, get_bdd100k_dataloader
    
    if DATA_DIR.exists():
        # Create dataset
        dataset = BDD100KDataset(DATA_DIR, split="train", image_size=(640, 640))
        print(f"Dataset size: {len(dataset)} images")
        
        if len(dataset) > 0:
            # Get a sample
            sample = dataset[0]
            print(f"\nSample keys: {sample.keys()}")
            print(f"Image shape: {sample['image'].shape}")
            print(f"Boxes shape: {sample['boxes'].shape}")
            print(f"Labels shape: {sample['labels'].shape}")
            print(f"Image ID: {sample['image_id']}")
            
            # Create DataLoader
            dataloader = get_bdd100k_dataloader(DATA_DIR, batch_size=4, num_workers=0, subset_size=100)
            batch = next(iter(dataloader))
            print(f"\nBatch keys: {batch.keys()}")
            print(f"Batch images shape: {batch['images'].shape}")
            print(f"Number of box tensors: {len(batch['boxes'])}")
        else:
            print("Dataset is empty. Run scripts/download_bdd100k.sh first.")
    else:
        print("Data directory not found. Run scripts/download_bdd100k.sh first.")

except ImportError as e:
    print(f"DataLoader dependencies not installed: {e}")
    print("Install with: pip install torch torchvision")
except Exception as e:
    print(f"DataLoader test failed: {e}")

## Test PyTorch DataLoader

Verify that the BDD100K PyTorch Dataset works correctly.

In [ ]:
# Test auto-labeler (requires ML dependencies: pip install ultralytics mobile-sam)
try:
    from data_engine.auto_labeler import AutoLabeler
    
    # Initialize auto-labeler (without SAM for faster testing)
    labeler = AutoLabeler(use_sam=False, conf_threshold=0.3)
    
    if image_files:
        test_image = image_files[0]
        print(f"Testing auto-labeler on: {test_image.name}")
        
        # Run auto-labeling
        results = labeler.label_image(test_image)
        
        print(f"\nDetected {len(results['boxes'])} objects:")
        for cls_name, conf, box in zip(results['class_names'], 
                                        results['confidences'], 
                                        results['boxes']):
            print(f"  - {cls_name}: {conf:.2%} at [{box[0]:.0f}, {box[1]:.0f}, {box[2]:.0f}, {box[3]:.0f}]")
        
        # Visualize results
        fig, axes = plt.subplots(1, 2, figsize=(16, 6))
        
        # Original with ground truth
        gt_labels = labels_by_name.get(test_image.stem, [])
        visualize_image_with_boxes(test_image, gt_labels, axes[0])
        axes[0].set_title(f"Ground Truth ({len(gt_labels)} objects)")
        
        # Auto-labeled
        img = Image.open(test_image)
        axes[1].imshow(img)
        axes[1].set_title(f"Auto-Labeled ({len(results['boxes'])} objects)")
        axes[1].axis('off')
        
        for box, cls_name, conf in zip(results['boxes'], results['class_names'], results['confidences']):
            x1, y1, x2, y2 = box
            rect = patches.Rectangle(
                (x1, y1), x2-x1, y2-y1,
                linewidth=2, edgecolor='lime', facecolor='none'
            )
            axes[1].add_patch(rect)
            axes[1].text(x1, y1-5, f"{cls_name} {conf:.0%}", fontsize=8, color='lime',
                        bbox=dict(boxstyle='round,pad=0.2', facecolor='black', alpha=0.7))
        
        plt.tight_layout()
        plt.show()
    else:
        print("No images available for testing.")

except ImportError as e:
    print(f"Auto-labeler dependencies not installed: {e}")
    print("Install with: pip install ultralytics")
except Exception as e:
    print(f"Auto-labeler test failed: {e}")

## Summary & Next Steps

### Dataset Statistics
- **BDD100K** is a large-scale driving dataset with 100K images
- Using a **5K subset** for efficient iteration during development
- Contains **10 object categories**: pedestrian, rider, car, truck, bus, train, motorcycle, bicycle, traffic light, traffic sign
- Average ~20 bounding boxes per image

### Data Pipeline Components
1. **BDD100KDataset**: PyTorch Dataset with automatic normalization and label parsing
2. **AutoLabeler**: YOLOv8 + MobileSAM for auto-generating labels on new images
3. **Ingestion Service**: Populates PostgreSQL from BDD100K files

### Next Steps
1. Download dataset: `./scripts/download_bdd100k.sh`
2. Start backend: `docker-compose up`
3. Ingest data via API or CLI
4. Proceed to model training: `02_model_training.ipynb`

In [ ]:
# Display sample images with annotations
if image_files and labels_by_name:
    # Find images that have labels
    labeled_images = [f for f in image_files if f.stem in labels_by_name][:6]
    
    if labeled_images:
        fig, axes = plt.subplots(2, 3, figsize=(18, 12))
        axes = axes.flatten()
        
        for ax, image_path in zip(axes, labeled_images):
            labels = labels_by_name.get(image_path.stem, [])
            visualize_image_with_boxes(image_path, labels, ax)
        
        plt.tight_layout()
        plt.show()
    else:
        print("No labeled images found.")
else:
    print("Images or labels not available. Run scripts/download_bdd100k.sh first.")

In [ ]:
# Color palette for different categories
CATEGORY_COLORS = {
    'car': '#FF6B6B',
    'truck': '#4ECDC4',
    'bus': '#45B7D1',
    'person': '#96CEB4',
    'pedestrian': '#96CEB4',
    'rider': '#FFEAA7',
    'bicycle': '#DDA0DD',
    'motorcycle': '#98D8C8',
    'traffic light': '#F7DC6F',
    'traffic sign': '#BB8FCE',
    'train': '#85C1E9',
}

def visualize_image_with_boxes(image_path, labels, ax=None):
    """Display an image with its bounding box annotations."""
    if ax is None:
        fig, ax = plt.subplots(1, 1, figsize=(12, 8))
    
    # Load and display image
    img = Image.open(image_path)
    ax.imshow(img)
    ax.set_title(image_path.name, fontsize=10)
    ax.axis('off')
    
    # Draw bounding boxes
    for label in labels:
        if "box2d" not in label:
            continue
            
        box = label["box2d"]
        category = label.get("category", "unknown")
        
        x1, y1 = box["x1"], box["y1"]
        x2, y2 = box["x2"], box["y2"]
        width, height = x2 - x1, y2 - y1
        
        color = CATEGORY_COLORS.get(category, '#FFFFFF')
        
        # Draw rectangle
        rect = patches.Rectangle(
            (x1, y1), width, height,
            linewidth=2, edgecolor=color, facecolor='none'
        )
        ax.add_patch(rect)
        
        # Add label
        ax.text(x1, y1 - 5, category, fontsize=8, color=color,
                bbox=dict(boxstyle='round,pad=0.2', facecolor='black', alpha=0.7))
    
    return ax

## Sample Images with Bounding Boxes

Visualize sample images with their ground truth annotations.

In [ ]:
# Plot class distribution
if category_counts:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Category distribution bar chart
    categories = list(category_counts.keys())
    counts = list(category_counts.values())
    
    ax1 = axes[0]
    bars = ax1.barh(categories, counts, color='steelblue')
    ax1.set_xlabel('Count')
    ax1.set_title('Object Category Distribution')
    ax1.invert_yaxis()
    
    # Add count labels
    for bar, count in zip(bars, counts):
        ax1.text(bar.get_width() + 100, bar.get_y() + bar.get_height()/2, 
                 f'{count:,}', va='center', fontsize=9)
    
    # Boxes per image histogram
    ax2 = axes[1]
    ax2.hist(boxes_per_image, bins=30, color='coral', edgecolor='black', alpha=0.7)
    ax2.set_xlabel('Number of Boxes')
    ax2.set_ylabel('Number of Images')
    ax2.set_title('Boxes per Image Distribution')
    ax2.axvline(np.mean(boxes_per_image), color='red', linestyle='--', 
                label=f'Mean: {np.mean(boxes_per_image):.1f}')
    ax2.legend()
    
    plt.tight_layout()
    plt.show()
else:
    print("No label data available for visualization.")

In [ ]:
# Count object categories
category_counts = Counter()
boxes_per_image = []

for item in labels_data:
    labels = item.get("labels", [])
    box_count = 0
    for label in labels:
        if "box2d" in label:
            category = label.get("category", "unknown")
            category_counts[category] += 1
            box_count += 1
    boxes_per_image.append(box_count)

print("Category distribution:")
for cat, count in category_counts.most_common():
    print(f"  {cat}: {count:,}")

print(f"\nTotal bounding boxes: {sum(category_counts.values()):,}")
print(f"Average boxes per image: {np.mean(boxes_per_image):.1f}")

# 01 - Data Exploration

Explore the BDD100K dataset for Constellation.

**Goals:**
- Load and visualize sample images
- Understand label format (COCO-style)
- Plot class distributions
- Compute dataset statistics

In [ ]:
import os
import sys
import json
from pathlib import Path
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

# Add project root to path for imports
PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

# Set data directory
DATA_DIR = PROJECT_ROOT / "data" / "bdd100k"
IMAGES_DIR = DATA_DIR / "images" / "100k" / "train"
LABELS_FILE = DATA_DIR / "labels" / "det_20" / "det_train.json"

print(f"Project root: {PROJECT_ROOT}")
print(f"Data directory: {DATA_DIR}")
print(f"Images directory: {IMAGES_DIR}")
print(f"Labels file: {LABELS_FILE}")
print()
print(f"Data exists: {DATA_DIR.exists()}")
print(f"Images exist: {IMAGES_DIR.exists()}")
print(f"Labels exist: {LABELS_FILE.exists()}")

In [ ]:
# Load labels if available
labels_data = []
labels_by_name = {}

if LABELS_FILE.exists():
    with open(LABELS_FILE, "r") as f:
        labels_data = json.load(f)
    labels_by_name = {Path(item["name"]).stem: item.get("labels", []) for item in labels_data}
    print(f"Loaded {len(labels_data)} image annotations")
else:
    print("Labels file not found. Run scripts/download_bdd100k.sh first.")

# Count images
image_files = list(IMAGES_DIR.glob("*.jpg")) if IMAGES_DIR.exists() else []
print(f"Found {len(image_files)} images")

## Class Distribution

Analyze the distribution of object categories in the dataset.

## Next Steps

After exploring the data, proceed to:
- `02_model_training.ipynb` - Train the HydraNet model